# 🍹 Sales Prediction Dataset Documentation

## 0. Quick Summary

### Inputs (Features)
- **DayType**  
- **Weather_8am_1pm**  
- **Weather_2pm_7pm**  
- **Weather_8pm_11pm**  
- **Traffic_Level**  
- **SchoolHoliday**  
- **PublicHoliday**  
- **Festival**  
- **BackToSchool**  
- **ExamSeason**  
- **PayDay**  
- **BigEventNearby**  
- **Promo_5PercentDiscount**  
- **Promo_10PercentDiscount**  
- **Promo_15PercentDiscount**  
- **Promo_20PercentDiscount**  
- **Promo_Buy1Free1**  
- **Promo_Buy2Free1**  
- **Promo_NewProductLaunch**  
- **Promo_AnniversarySale**  
- **Promo_SocialMediaCampaign**

### Outputs (Targets)
- **Drink_IcedCaramelMacchiato**  
- **Drink_MochaFrappe**  
- **Drink_StrawberrySmoothie**  
- **Drink_MangoPassionfruitCooler**  
- **Drink_MatchaLatte**  
- **Drink_ChocolateHazelnutShake**  
- **Drink_BlueberryLemonadeSparkler**  
- **Drink_AvocadoCoffee**

*(All targets are integer counts = cups sold)*

---

## 1. Data Collection & Storage

Each record should represent **one day** (or per shift/hour if higher frequency is needed).

### Columns

- **Date (ISO format)**  
- **Day (e.g., Monday, Tuesday...)**  
- **DayType** → Weekday / Weekend  
- **Weather_8am_1pm, Weather_2pm_7pm, Weather_8pm_11pm**  
  - *Categorical: Sunny / Partly Cloudy / Cloudy / Light Rain / Heavy Rain / Thunderstorm*  
- **Traffic_Level** → Low / Medium / High  

### Binary Flags (0/1)
- **SchoolHoliday**  
- **PublicHoliday**  
- **Festival**  
- **BackToSchool**  
- **ExamSeason**  
- **PayDay**  
- **BigEventNearby**

### Promo Flags (0/1)
- **Promo_5PercentDiscount**  
- **Promo_10PercentDiscount**  
- **Promo_15PercentDiscount**  
- **Promo_20PercentDiscount**  
- **Promo_Buy1Free1**  
- **Promo_Buy2Free1**  
- **Promo_NewProductLaunch**  
- **Promo_AnniversarySale**  
- **Promo_SocialMediaCampaign**

### Targets (Cups Sold, Integers)
- **Drink_IcedCaramelMacchiato**  
- **Drink_MochaFrappe**  
- **Drink_StrawberrySmoothie**  
- **Drink_MangoPassionfruitCooler**  
- **Drink_MatchaLatte**  
- **Drink_ChocolateHazelnutShake**  
- **Drink_BlueberryLemonadeSparkler**  
- **Drink_AvocadoCoffee**

### Optional
- **Total_Sales** → Auto (sum of all drinks)  
- **Notes** → Free text (events, special observations, anomalies, etc.)  


### Fetch and Anylyze Data

In [ ]:
import pandas as pd

sales_df = pd.read_excel("./dataset/sales2.xlsx", "Sale1")
# sales_df

print("\n📄 Info tentang dataset:")
print(sales_df.info())

print("\n📈 Statistik ringkas untuk numeric columns:")
display(sales_df.describe())

### Data Preperation

In [ ]:
# check missing value
print("missing value: ", sales_df.isnull().sum())

# separate the data
feature_cols = [
    "DayType", "Weather_8am_1pm", "Weather_2pm_7pm", "Weather_8pm_11pm",
    "Traffic_Level", "SchoolHoliday", "PublicHoliday", "Festival",
    "BackToSchool", "ExamSeason", "PayDay", "BigEventNearby",
    "Promo_5PercentDiscount", "Promo_10PercentDiscount",
    "Promo_15PercentDiscount", "Promo_20PercentDiscount",
    "Promo_Buy1Free1", "Promo_Buy2Free1", "Promo_NewProductLaunch",
    "Promo_AnniversarySale", "Promo_SocialMediaCampaign"
]

# Outputs (the 8 drinks)
target_cols = [
    "Drink_IcedCaramelMacchiato", "Drink_MochaFrappe",
    "Drink_StrawberrySmoothie", "Drink_MangoPassionfruitCooler",
    "Drink_MatchaLatte", "Drink_ChocolateHazelnutShake",
    "Drink_BlueberryLemonadeSparkler", "Drink_AvocadoCoffee"
]

X = sales_df[feature_cols]
y = sales_df[target_cols]

# indentify category and numerical
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

### Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

drink = y.columns.tolist()
promo_cols = [
    "Promo_5PercentDiscount", "Promo_10PercentDiscount", "Promo_15PercentDiscount",
    "Promo_20PercentDiscount", "Promo_Buy1Free1", "Promo_Buy2Free1",
    "Promo_NewProductLaunch", "Promo_AnniversarySale", "Promo_SocialMediaCampaign"
]

# chart 1: histogram jualan setiap air
plt.figure(figsize=(16, 8))
for i, drink in enumerate(drink):
    plt.subplot(2, 4, i+1)
    sns.histplot(y[drink], bins=20, kde=True, color='skyblue')
    plt.title(f"Histogram jualan: {drink}", fontsize=10)
    plt.xlabel("Jumlah Cawan terjual", fontsize=10)
    plt.ylabel("Bilangan Hari", fontsize=8)
plt.suptitle("Distribusi Jualan Setiap Minuman", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()



### Spiliting The Data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

### Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# perlu handle categorical & numerical features secara sistematik.
# - Numerical features → scaling (optional untuk GBDT, tapi bagus kalau nak consistent).
# - Categorical features → one-hot encoding.

# Semua digabung dengan ColumnTransformer supaya pipeline kemas & modular.

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)


### Model Training

In [ ]:
# Step 5: Model Training (GBDT, RandomForest, XGBoost, LightGBM, CatBoost)

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline

# Simpan model dalam dictionary
models = {
    "RandomForest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", MultiOutputRegressor(RandomForestRegressor(
            n_estimators=200, random_state=42, n_jobs=-1)))
    ]),
    
    "GradientBoosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", MultiOutputRegressor(GradientBoostingRegressor(
            random_state=42)))
    ]),
    
    "XGBoost": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", MultiOutputRegressor(XGBRegressor(
            random_state=42, n_estimators=200, n_jobs=-1)))
    ]),
    
    "LightGBM": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", MultiOutputRegressor(LGBMRegressor(
            random_state=42, n_estimators=200, n_jobs=-1)))
    ]),
    
    "CatBoost": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", MultiOutputRegressor(CatBoostRegressor(
            random_state=42, verbose=0)))
    ])
}

# Train semua model
trained_models = {}
for name, model in models.items():
    print(f"🔄 Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model

print("\n✅ Semua model telah siap dilatih!")


In [ ]:
# Step 6: Model Evaluation

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# Dictionary untuk simpan result setiap model
evaluation_results = {}

# Loop melalui semua trained model
for name, model in trained_models.items():
    print(f"🔎 Evaluating {name}...")
    
    # Predict untuk test set
    y_pred = model.predict(X_test)
    
    # Kira metric
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    
    # Simpan result
    evaluation_results[name] = {
        "MAE": round(mae, 2),
        "MSE": round(mse, 2),
        "R2": round(r2, 2)
    }

# Convert ke DataFrame supaya mudah compare
results_df = pd.DataFrame(evaluation_results).T
results_df = results_df.sort_values(by="R2", ascending=False)

results_df


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Define parameter grid for Gradient Boosting
param_grid = {
    "regressor__estimator__n_estimators": [100, 200, 300],
    "regressor__estimator__learning_rate": [0.01, 0.05, 0.1],
    "regressor__estimator__max_depth": [3, 5, 7],
    "regressor__estimator__subsample": [0.8, 1.0]
}

# Ambil pipeline GradientBoosting dari dictionary
gb_pipeline = models["GradientBoosting"]

# GridSearch dengan 3-fold CV
grid_search = GridSearchCV(
    gb_pipeline,
    param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

print("🔍 Running GridSearch for GradientBoosting...")
grid_search.fit(X_train, y_train)

# Best model
best_gb = grid_search.best_estimator_
print("✅ Best Parameters:", grid_search.best_params_)

# Evaluate di test set
y_pred = best_gb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n📊 GradientBoosting Best Model Evaluation:")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R2 : {r2:.2f}")


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline

# Best parameters from GridSearch
best_params = {
    "learning_rate": 0.1,
    "max_depth": 3,
    "n_estimators": 100,
    "subsample": 0.8
}

# Build a new pipeline with best params
best_gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", MultiOutputRegressor(
        GradientBoostingRegressor(
            learning_rate=best_params["learning_rate"],
            max_depth=best_params["max_depth"],
            n_estimators=best_params["n_estimators"],
            subsample=best_params["subsample"],
            random_state=42
        )
    ))
])

# Retrain on training data
print("🔄 Retraining GradientBoosting with best parameters...")
best_gb_model.fit(X_train, y_train)

# Evaluate on test data
y_pred = best_gb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n📊 Final GradientBoosting Model Evaluation (Best Params):")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R2 : {r2:.2f}")

# (Optional) Simpan dalam trained_models dictionary
trained_models["GradientBoosting_Best"] = best_gb_model


In [ ]:
fig, axes = plt.subplots(ncols=3, nrows=1, constrained_layout=True, figsize=(16, 5))
plt.figure(figsize=(16, 8))

axes[0].bar(results_df.index, results_df['MSE'])
axes[0].set_title('Mean Squared Error')
axes[0].set_xlabel('Models')
axes[0].set_ylabel('MSE')
axes[0].grid(axis='y', linestyle='--')

axes[1].bar(results_df.index, results_df['MAE'])
axes[1].set_title('Mean Absolute Error')
axes[1].set_xlabel('Models')
axes[1].set_ylabel('MAE')
axes[1].grid(axis='y', linestyle='--')


axes[2].bar(results_df.index, results_df['R2'])
axes[2].set_title('R² Score')
axes[2].set_xlabel('Models')
axes[2].set_ylabel('R²')
axes[2].grid(axis='y', linestyle='--')


# Rotate labels
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.show()

In [ ]:
# Interaction features
X = X.copy()   # pastikan buat salinan dulu untuk elak warning

X.loc[:, "Promo20_Festival"] = X["Promo_20PercentDiscount"] * X["Festival"]
X.loc[:, "PayDay_B1F1"] = X["PayDay"] * X["Promo_Buy1Free1"]


import numpy as np

y_log = np.log1p(y)   # log(1+y) untuk elak log(0)


In [ ]:
# Interaction features
X["Promo20_Festival"] = X["Promo_20PercentDiscount"] * X["Festival"]
X["PayDay_B1F1"] = X["PayDay"] * X["Promo_Buy1Free1"]


from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline


model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(random_state=42))
])

param_grid = {
    "regressor__n_estimators": [100, 200, 300],
    "regressor__learning_rate": [0.05, 0.1, 0.2],
    "regressor__max_depth": [3, 5, 7]
}

grid_search = GridSearchCV(
    model,
    param_grid,
    scoring="r2",
    cv=3,
    n_jobs=-1
)

grid_search.fit(X_train, y_train.iloc[:, 0])
print("Best Params:", grid_search.best_params_)




In [ ]:
sales_df["Drink_IcedCaramelMacchiato_lag1"] = sales_df["Drink_IcedCaramelMacchiato"].shift(1)
sales_df["Drink_IcedCaramelMacchiato_lag7"] = sales_df["Drink_IcedCaramelMacchiato"].shift(7)


In [ ]:
best_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=3,
        n_estimators=100,
        random_state=42
    ))
])

best_model.fit(X_train, y_train.iloc[:, 0])  # contoh untuk 1 drink


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = best_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test.iloc[:, 0], y_pred))
print("MSE:", mean_squared_error(y_test.iloc[:, 0], y_pred))
print("R2 :", r2_score(y_test.iloc[:, 0], y_pred))


In [ ]:
from sklearn.multioutput import MultiOutputRegressor

multi_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", MultiOutputRegressor(
        GradientBoostingRegressor(
            learning_rate=0.05,
            max_depth=3,
            n_estimators=100,
            random_state=42
        )
    ))
])

multi_model.fit(X_train, y_train)   # semua drinks sekali
y_pred_multi = multi_model.predict(X_test)


In [ ]:
for i, col in enumerate(y_train.columns):
    mae = mean_absolute_error(y_test.iloc[:, i], y_pred_multi[:, i])
    r2 = r2_score(y_test.iloc[:, i], y_pred_multi[:, i])
    print(f"{col}: MAE={mae:.2f}, R2={r2:.2f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =============================
# 1. Copy dataset asal (X, y)
# =============================
X = X.copy()
y = y.copy()

# =============================
# 2. Tambah Lag & Rolling Features
# =============================
for col in y.columns:
    for lag in [1, 2, 3, 7]:  # 1,2,3,7 hari lepas
        X[f"{col}_lag{lag}"] = y[col].shift(lag)
    X[f"{col}_rolling7"] = y[col].shift(1).rolling(7).mean()

# Drop rows dengan NaN (akibat shift/rolling)
X = X.dropna()
y = y.loc[X.index]

# =============================
# 3. Train-Test Split
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # time-series → jangan shuffle
)

# =============================
# 4. Preprocessor
# =============================
numerical_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# =============================
# 5. Model GradientBoosting dengan Best Params
# =============================
gbr_best = GradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=3,
    n_estimators=100,
    random_state=42
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", MultiOutputRegressor(gbr_best))
])

# =============================
# 6. Train Model
# =============================
model.fit(X_train, y_train)

# =============================
# 7. Evaluation
# =============================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("📊 Model Evaluation with Lag & Rolling Features")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R2 : {r2:.2f}")
